In [2]:
import numpy as np
from pathlib import Path

In [4]:
# Processed data path

data_dir = Path("processed_data")

print("Processed data directory:", data_dir.resolve())
print("Directory exists:", data_dir.exists())

Processed data directory: E:\data of brain projec\data of brain project\EEG_project\processed_data
Directory exists: True


In [5]:
alpha_asymmetry = np.load(data_dir / "alpha_asymmetry.npy")
alpha_power = np.load(data_dir / "alpha_power.npy")

beta_asymmetry = np.load(data_dir / "beta_asymmetry.npy")
beta_power = np.load(data_dir / "beta_power.npy")

correlation = np.load(data_dir / "correlation.npy")
mi = np.load(data_dir / "mi.npy")

print("Alpha Asymmetry:", alpha_asymmetry.shape)
print("Alpha Power:", alpha_power.shape)
print("Beta Asymmetry:", beta_asymmetry.shape)
print("Beta Power:", beta_power.shape)
print("Correlation:", correlation.shape)
print("MI:", mi.shape)

Alpha Asymmetry: (85330, 7)
Alpha Power: (85330, 14)
Beta Asymmetry: (85330, 7)
Beta Power: (85330, 14)
Correlation: (85330, 14, 14)
MI: (85330, 14, 14)


In [ ]:
# Check NaN and Inf values
features = {
    "Alpha Asymmetry": alpha_asymmetry,
    "Alpha Power": alpha_power,
    "Beta Asymmetry": beta_asymmetry,
    "Beta Power": beta_power,
    "Correlation": correlation,
    "MI": mi
}

for name, feature in features.items():
    print(
        f"{name}: "
        f"NaN = {np.isnan(feature).sum()}, "
        f"Inf = {np.isinf(feature).sum()}"
    )

Alpha Asymmetry: NaN = 0, Inf = 0
Alpha Power: NaN = 0, Inf = 0
Beta Asymmetry: NaN = 0, Inf = 0
Beta Power: NaN = 0, Inf = 0
Correlation: NaN = 0, Inf = 0
MI: NaN = 0, Inf = 0


In [ ]:
# Check feature range
for name, feature in features.items():
    print(f"\n{name}")
    print("  Min :", np.min(feature))
    print("  Max :", np.max(feature))
    print("  Mean:", np.mean(feature))
    print("  Std :", np.std(feature))


Alpha Asymmetry
  Min : -24.94967
  Max : 24.894274
  Mean: -1.3043098
  Std : 1.674336

Alpha Power
  Min : 4.5528534e-26
  Max : 662052.75
  Mean: 139.82608
  Std : 3339.3447

Beta Asymmetry
  Min : -25.397486
  Max : 26.211737
  Mean: -1.0205685
  Std : 1.5198295

Beta Power
  Min : 7.737298e-29
  Max : 663713.4
  Mean: 5370.5796
  Std : 32078.121

Correlation
  Min : -0.9934373
  Max : 1.0
  Mean: 0.44027072
  Std : 0.3680898

MI
  Min : 0.0
  Max : 3.9112108
  Mean: 0.29891795
  Std : 0.40816706


In [8]:
# ============================================================
# Flatten matrix-based features
# ============================================================

correlation_flat = correlation.reshape(
    correlation.shape[0], -1
)

mi_flat = mi.reshape(
    mi.shape[0], -1
)

print("Original Correlation shape:", correlation.shape)
print("Flattened Correlation shape:", correlation_flat.shape)

print("\nOriginal MI shape:", mi.shape)
print("Flattened MI shape:", mi_flat.shape)

Original Correlation shape: (85330, 14, 14)
Flattened Correlation shape: (85330, 196)

Original MI shape: (85330, 14, 14)
Flattened MI shape: (85330, 196)


In [ ]:
# Extract unique connections from symmetric matrices

n_channels = correlation.shape[1]

upper_triangle = np.triu_indices(
    n_channels,
    k=1
)

correlation_features = correlation[
    :, upper_triangle[0], upper_triangle[1]
]

mi_features = mi[
    :, upper_triangle[0], upper_triangle[1]
]

print("Correlation features shape:", correlation_features.shape)
print("MI features shape:", mi_features.shape)

Correlation features shape: (85330, 91)
MI features shape: (85330, 91)


In [ ]:
# Check prepared feature statistics

prepared_features = {
    "Alpha Power": alpha_power,
    "Beta Power": beta_power,
    "Alpha Asymmetry": alpha_asymmetry,
    "Beta Asymmetry": beta_asymmetry,
    "Correlation": correlation_features,
    "MI": mi_features
}

for name, feature in prepared_features.items():
    print(
        f"{name}: "
        f"mean={np.mean(feature):.6f}, "
        f"std={np.std(feature):.6f}, "
        f"min={np.min(feature):.6f}, "
        f"max={np.max(feature):.6f}"
    )

Alpha Power: mean=139.826080, std=3339.344727, min=0.000000, max=662052.750000
Beta Power: mean=5370.579590, std=32078.121094, min=0.000000, max=663713.375000
Alpha Asymmetry: mean=-1.304310, std=1.674336, min=-24.949671, max=24.894274
Beta Asymmetry: mean=-1.020568, std=1.519830, min=-25.397486, max=26.211737
Correlation: mean=0.397215, std=0.346351, min=-0.993437, max=0.999986
MI: mean=0.321912, std=0.414745, min=0.000000, max=3.911211


In [11]:
# ============================================================
# Check Power distribution
# ============================================================

print("Alpha Power percentiles:")
print(np.percentile(
    alpha_power,
    [0, 25, 50, 75, 90, 95, 99, 99.9, 100]
))

print("\nBeta Power percentiles:")
print(np.percentile(
    beta_power,
    [0, 25, 50, 75, 90, 95, 99, 99.9, 100]
))

Alpha Power percentiles:
[4.55285340e-26 4.55518329e+00 1.13160276e+01 2.52451339e+01
 5.80737625e+01 1.19456822e+02 2.15140893e+03 1.61931090e+04
 6.62052750e+05]

Beta Power percentiles:
[7.73729776e-29 7.96064579e+00 1.62444830e+01 3.80014620e+01
 3.04047464e+02 1.58970783e+04 2.37914950e+05 3.06345897e+05
 6.63713375e+05]


In [12]:
# ============================================================
# Log transform Power features
# ============================================================

alpha_power_log = np.log1p(alpha_power)
beta_power_log = np.log1p(beta_power)

print("Alpha Power - before:")
print(
    "Mean:", np.mean(alpha_power),
    "Std:", np.std(alpha_power),
    "Max:", np.max(alpha_power)
)

print("\nAlpha Power - after log1p:")
print(
    "Mean:", np.mean(alpha_power_log),
    "Std:", np.std(alpha_power_log),
    "Max:", np.max(alpha_power_log)
)

print("\nBeta Power - before:")
print(
    "Mean:", np.mean(beta_power),
    "Std:", np.std(beta_power),
    "Max:", np.max(beta_power)
)

print("\nBeta Power - after log1p:")
print(
    "Mean:", np.mean(beta_power_log),
    "Std:", np.std(beta_power_log),
    "Max:", np.max(beta_power_log)
)

Alpha Power - before:
Mean: 139.82608 Std: 3339.3447 Max: 662052.75

Alpha Power - after log1p:
Mean: 2.5393171 Std: 1.4529135 Max: 13.403102

Beta Power - before:
Mean: 5370.5796 Std: 32078.121 Max: 663713.4

Beta Power - after log1p:
Mean: 3.343439 Std: 2.309033 Max: 13.405607


In [ ]:
# Standardize prepared features

from sklearn.preprocessing import StandardScaler

scalers = {}

scaled_features = {}

for name, feature in {
    "Alpha Power": alpha_power_log,
    "Beta Power": beta_power_log,
    "Alpha Asymmetry": alpha_asymmetry,
    "Beta Asymmetry": beta_asymmetry,
    "Correlation": correlation_features,
    "MI": mi_features
}.items():

    scaler = StandardScaler()

    scaled = scaler.fit_transform(feature)

    scalers[name] = scaler
    scaled_features[name] = scaled.astype(np.float32)

    print(
        f"{name}: "
        f"shape={scaled_features[name].shape}, "
        f"mean={np.mean(scaled_features[name]):.6f}, "
        f"std={np.std(scaled_features[name]):.6f}"
    )

Alpha Power: shape=(85330, 14), mean=-0.000000, std=1.000000
Beta Power: shape=(85330, 14), mean=-0.000000, std=1.000000
Alpha Asymmetry: shape=(85330, 7), mean=0.000000, std=1.000000
Beta Asymmetry: shape=(85330, 7), mean=0.000000, std=1.000000
Correlation: shape=(85330, 91), mean=-0.000000, std=1.000000
MI: shape=(85330, 91), mean=-0.000000, std=1.000000
